In [2]:
interface QualichargeEVSEDynamic {
  etat_pdc: string;
  horodatage: string;
  occupation_pdc: string;
  etat_prise_type_2: string;
  id_pdc_itinerance: string;
  etat_prise_type_ef: string;
  etat_prise_type_chademo: string;
  etat_prise_type_combo_ccs: string;
}

type QualichargeMerged = QualichargeEVSEStatique & {
  dynamic?: QualichargeEVSEDynamic;
};

In [ ]:
import {
  asyncBufferFromUrl,
  parquetReadObjects,
} from "hyparquet";
import { compressors } from "hyparquet-compressors";

// URLs
const STATIC_URL =
  "https://object.files.data.gouv.fr/hydra-parquet/hydra-parquet/8bb0a6e2-1016-42ba-aaee-f72f55c82e9f.parquet";

const DYNAMIC_URL =
  "https://object.files.data.gouv.fr/hydra-parquet/hydra-parquet/411443b1-6667-473f-8217-1c57c167408f.parquet";

// clé composite
function key(
  id_station_itinerance: string,
  id_pdc_itinerance: string
) {
  return `${id_pdc_itinerance}`;
//   return `${id_station_itinerance}__${id_pdc_itinerance}`;
}

async function main() {
  // --- 1. Load dynamique ---
  const dynamicFile = await asyncBufferFromUrl({ url: DYNAMIC_URL });

  const dynamicRows = (await parquetReadObjects({
    file: dynamicFile,
    compressors,
  })) as QualichargeEVSEDynamic[];

  // index
  const dynamicMap = new Map<string, QualichargeEVSEDynamic>();

  for (const row of dynamicRows) {
    dynamicMap.set(
      key(
        // ⚠️ pas dans ton schema mais présent dans dataset réel
        (row as any).id_station_itinerance ?? "",
        row.id_pdc_itinerance
      ),
      row
    );
  }

  // --- 2. Load statique ---
  const staticFile = await asyncBufferFromUrl({ url: STATIC_URL });

  const staticRows = (await parquetReadObjects({
    file: staticFile,
    compressors,
  })) as QualichargeEVSEStatique[];

  // --- 3. Merge ---
  const merged: QualichargeMerged[] = [];

  for (const row of staticRows) {
    const k = key(row.id_station_itinerance, row.id_pdc_itinerance);

    merged.push({
      ...row,
      dynamic: dynamicMap.get(k),
    });
  }

  // console.log(merged.slice(0, 5));

  const map = new Map<string, number>();

for (const row of dynamicRows) {
  const k = row.id_pdc_itinerance;
  map.set(k, (map.get(k) ?? 0) + 1);
}

const multi = [...map.entries()].filter(([, c]) => c > 1);

console.log("PDC avec plusieurs états:", multi.length);
console.log(multi.slice(0, 10));
  return merged;
}

main();

Promise { <pending> }

PDC avec plusieurs états: 0
[]


In [ ]:
import {
  asyncBufferFromUrl,
  parquetReadObjects,
} from "hyparquet";
import { compressors } from "hyparquet-compressors";

const STATIC_URL =
  "https://object.files.data.gouv.fr/hydra-parquet/hydra-parquet/8bb0a6e2-1016-42ba-aaee-f72f55c82e9f.parquet";

const DYNAMIC_URL =
  "https://object.files.data.gouv.fr/hydra-parquet/hydra-parquet/411443b1-6667-473f-8217-1c57c167408f.parquet";

async function main() {
  // --- load dynamic
  const dynamicFile = await asyncBufferFromUrl({ url: DYNAMIC_URL });
  const dynamicRows = await parquetReadObjects({
    file: dynamicFile,
    compressors,
  }) as any[];

  const dynamicSet = new Set(
    dynamicRows.map(r => r.id_pdc_itinerance)
  );

  // --- load static
  const staticFile = await asyncBufferFromUrl({ url: STATIC_URL });
  const staticRows = await parquetReadObjects({
    file: staticFile,
    compressors,
  }) as any[];

  // --- diff
  const missingDynamic = staticRows.filter(
    r => !dynamicSet.has(r.id_pdc_itinerance)
  );

  console.log("---- RESULTS ----");
  console.log("Total static:", staticRows.length);
  console.log("Total dynamic:", dynamicRows.length);
  console.log("PDC sans état dynamique:", missingDynamic.length);

  console.log("\nSample:");
  console.log(missingDynamic.slice(0, 5));
}

main();

Promise { <pending> }

---- RESULTS ----
Total static: 63120
Total dynamic: 60203
PDC sans état dynamique: 2955

Sample:
[
  {
    nom_amenageur: "R3 SPV COM",
    siren_amenageur: "949415814",
    contact_amenageur: "avanoost@r3-charge.fr",
    nom_operateur: "R3",
    contact_operateur: "avanoost@r3-charge.fr",
    telephone_operateur: "tel:+33-1-23-45-67-89",
    nom_enseigne: "R3 SPV COM",
    id_station_itinerance: "FRR3MP1064050",
    id_station_local: "FR*R3M*1064050",
    nom_station: "Hersin-Coupigny - ALDI",
    implantation_station: "Parking privé à usage public",
    adresse_station: "Rue Victor Hugo 73, 62530 Hersin-Coupigny",
    code_insee_commune: "62443",
    coordonneesXY: "[2.637454787, 50.451552637]",
    nbre_pdc: 5n,
    id_pdc_itinerance: "FRR3ME5373846",
    id_pdc_local: "FR*R3M*E5373846",
    puissance_nominale: 180,
    prise_type_ef: false,
    prise_type_2: false,
    prise_type_combo_ccs: true,
    prise_type_chademo: false,
    prise_type_autre: false,
    gratuit: null,
    pa